<a href="https://colab.research.google.com/github/hetalsharmaaa/QuickBrief/blob/main/digitalrecognizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install tensorflow scikit-learn matplotlib seaborn pandas numpy --break-system-packages -q 2>&1 | tail -5

In [2]:
import tensorflow as tf; import sklearn; print('TF:', tf.__version__, '| sklearn:', sklearn.__version__)

TF: 2.20.0 | sklearn: 1.6.1


In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("  PROJECT 1: Handwritten Digit Recognizer (CNN + MNIST)")
print("=" * 60)

  PROJECT 1: Handwritten Digit Recognizer (CNN + MNIST)


In [4]:
print("\n[1/6] Loading MNIST Dataset...")
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"  Training samples : {X_train.shape[0]:,}")
print(f"  Testing  samples : {X_test.shape[0]:,}")
print(f"  Image dimensions : {X_train.shape[1]}×{X_train.shape[2]} pixels (grayscale)")
print(f"  Pixel range      : {X_train.min()} – {X_train.max()}")
print(f"  Classes          : {np.unique(y_train).tolist()}  (digits 0–9)")


[1/6] Loading MNIST Dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Training samples : 60,000
  Testing  samples : 10,000
  Image dimensions : 28×28 pixels (grayscale)
  Pixel range      : 0 – 255
  Classes          : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  (digits 0–9)


In [5]:
print("\n[2/6] Preprocessing Data...")

# Normalize pixel values from [0,255] → [0,1]
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32")  / 255.0

# Reshape for CNN: add channel dimension (28,28) → (28,28,1)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

print(f"  Normalized pixel range : 0.0 – 1.0")
print(f"  Input shape for CNN    : {X_train.shape[1:]}")



[2/6] Preprocessing Data...
  Normalized pixel range : 0.0 – 1.0
  Input shape for CNN    : (28, 28, 1)


In [6]:
print("\n[3/6] Building CNN Architecture...")

model = keras.Sequential([
    # Block 1 – Feature Extraction
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  input_shape=(28, 28, 1), name='Conv_Block1_1'),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  name='Conv_Block1_2'),
    layers.MaxPooling2D((2, 2), name='MaxPool_1'),
    layers.Dropout(0.25, name='Dropout_1'),

    # Block 2 – Deeper Feature Extraction
    layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                  name='Conv_Block2_1'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                  name='Conv_Block2_2'),
    layers.MaxPooling2D((2, 2), name='MaxPool_2'),
    layers.Dropout(0.25, name='Dropout_2'),

    # Classifier Head
    layers.Flatten(name='Flatten'),
    layers.Dense(512, activation='relu', name='Dense_1'),
    layers.Dropout(0.5, name='Dropout_3'),
    layers.Dense(10, activation='softmax', name='Output')
], name='Digit_Recognizer_CNN')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

total_params = model.count_params()
print(f"\n  Total trainable parameters: {total_params:,}")



[3/6] Building CNN Architecture...


Model: "Digit_Recognizer_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Conv_Block1_1 (Conv2D)          │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv_Block1_2 (Conv2D)          │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool_1 (MaxPooling2D)        │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_1 (Dropout)             │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv_Block2_1 (Conv2D)          │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv_Block2_2 (Conv2D)          │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool_2 (MaxPooling2D)        │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_2 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_1 (Dense)                 │ (None, 512)            │     1,606,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,676,266 (6.39 MB)

 Trainable params: 1,676,266 (6.39 MB)

 Non-trainable params: 0 (0.00 B)


  Total trainable parameters: 1,676,266


In [9]:
print("\n[4/6] Training the CNN...")

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-6, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)



[4/6] Training the CNN...
Epoch 1/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 263s 623ms/step - accuracy: 0.9662 - loss: 0.1108 - val_accuracy: 0.9867 - val_loss: 0.0412 - learning_rate: 0.0010
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 258s 612ms/step - accuracy: 0.9809 - loss: 0.0626 - val_accuracy: 0.9907 - val_loss: 0.0317 - learning_rate: 0.0010
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 253s 592ms/step - accuracy: 0.9857 - loss: 0.0460 - val_accuracy: 0.9905 - val_loss: 0.0330 - learning_rate: 0.0010
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 591ms/step - accuracy: 0.9890 - loss: 0.0360
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
422/422 ━━━━━━━━━━━━━━━━━━━━ 267s 605ms/step - accuracy: 0.9892 - loss: 0.0363 - val_accuracy: 0.9898 - val_loss: 0.0366 - learning_rate: 0.0010
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 259s 613ms/step - accuracy: 0.9921 - loss: 0.0251 - val_accuracy: 0.9932 - val_loss: 0.0233 - learning_rate: 5.0000e-04
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━

In [10]:
print("\n[5/6] Evaluating on Test Set...")

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"  Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")

y_pred_probs = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

print("\n  Per-class Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=[str(i) for i in range(10)]))


[5/6] Evaluating on Test Set...
  Test Accuracy : 99.40%
  Test Loss     : 0.0179

  Per-class Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       980
           1       0.99      1.00      0.99      1135
           2       1.00      1.00      1.00      1032
           3       1.00      1.00      1.00      1010
           4       1.00      0.99      0.99       982
           5       0.99      0.99      0.99       892
           6       1.00      0.99      0.99       958
           7       0.99      0.99      0.99      1028
           8       0.99      0.99      0.99       974
           9       0.99      0.99      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [13]:
print("\n[6/6] Generating Visualizations...")

fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('#0d0d0d')

# ── Panel A: Sample MNIST Images ──────────────────────────────
ax_title = fig.add_subplot(6, 1, 1)
ax_title.axis('off')
ax_title.text(0.5, 0.5,
    'PROJECT 3 — Handwritten Digit Recognizer (CNN + MNIST)',
    ha='center', va='center', fontsize=18, fontweight='bold',
    color='white',
    fontfamily='DejaVu Sans')
ax_title.text(0.5, 0.0,
    f'Test Accuracy: {test_accuracy*100:.2f}%  |  Parameters: {total_params:,}',
    ha='center', va='center', fontsize=13, color='#00e5ff')

# ── Panel B: Sample Digits ──────────────────────────────────
gs = gridspec.GridSpec(5, 1, figure=fig, hspace=0.5, top=0.88, bottom=0.04)

gs_digits = gridspec.GridSpecFromSubplotSpec(2, 10, subplot_spec=gs[0], wspace=0.1, hspace=0.5)
fig.text(0.5, 0.855, '① Sample Training Images (one per class)',
         ha='center', fontsize=13, color='#ffd54f', fontweight='bold')

# Show one sample per digit
for digit in range(10):
    idx = np.where(y_test == digit)[0][0]
    img = X_test[idx].reshape(28, 28)
    ax = fig.add_subplot(gs_digits[0, digit])
    ax.imshow(img, cmap='plasma')
    ax.set_title(f'{digit}', color='white', fontsize=11, pad=2)
    ax.axis('off')

# Show correct predictions
for i in range(10):
    correct_mask = (y_test == y_pred) & (y_test == i)
    idx = np.where(correct_mask)[0][0]
    img = X_test[idx].reshape(28, 28)
    ax = fig.add_subplot(gs_digits[1, i])
    ax.imshow(img, cmap='viridis')
    ax.set_title(f'✓{i}', color='#69f0ae', fontsize=10, pad=2)
    ax.axis('off')

# ── Panel C: Training History ──────────────────────────────
gs_history = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[1], wspace=0.3)
fig.text(0.5, 0.68, '② Training History', ha='center',
         fontsize=13, color='#ffd54f', fontweight='bold')

epochs_ran = range(1, len(history.history['accuracy']) + 1)

ax_acc = fig.add_subplot(gs_history[0])
ax_acc.set_facecolor('#1a1a2e')
ax_acc.plot(epochs_ran, history.history['accuracy'],
            color='#00e5ff', linewidth=2.5, marker='o', markersize=5, label='Train Acc')
ax_acc.plot(epochs_ran, history.history['val_accuracy'],
            color='#ff4081', linewidth=2.5, marker='s', markersize=5, label='Val Acc')
ax_acc.axhline(y=test_accuracy, color='#69f0ae', linestyle='--', linewidth=1.5,
               label=f'Test Acc={test_accuracy:.4f}')
ax_acc.set_title('Model Accuracy', color='white', fontsize=12)
ax_acc.set_xlabel('Epoch', color='#aaa')
ax_acc.set_ylabel('Accuracy', color='#aaa')
ax_acc.tick_params(colors='#aaa')
ax_acc.legend(facecolor='#2d2d2d', labelcolor='white', fontsize=9)
ax_acc.set_ylim(0.9, 1.01)
for spine in ax_acc.spines.values():
    spine.set_color('#444')

ax_loss = fig.add_subplot(gs_history[1])
ax_loss.set_facecolor('#1a1a2e')
ax_loss.plot(epochs_ran, history.history['loss'],
             color='#ffab40', linewidth=2.5, marker='o', markersize=5, label='Train Loss')
ax_loss.plot(epochs_ran, history.history['val_loss'],
             color='#ea80fc', linewidth=2.5, marker='s', markersize=5, label='Val Loss')
ax_loss.set_title('Model Loss', color='white', fontsize=12)
ax_loss.set_xlabel('Epoch', color='#aaa')
ax_loss.set_ylabel('Loss', color='#aaa')
ax_loss.tick_params(colors='#aaa')
ax_loss.legend(facecolor='#2d2d2d', labelcolor='white', fontsize=9)
for spine in ax_loss.spines.values():
    spine.set_color('#444')

# ── Panel D: Confusion Matrix ──────────────────────────────
ax_cm = fig.add_subplot(gs[2])
ax_cm.set_facecolor('#1a1a2e')
fig.text(0.5, 0.495, '③ Confusion Matrix on Test Set',
         ha='center', fontsize=13, color='#ffd54f', fontweight='bold')

cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=ax_cm, cbar_kws={'label': 'Recall %'},
            linewidths=0.5, linecolor='#333',
            xticklabels=range(10), yticklabels=range(10))
ax_cm.set_xlabel('Predicted Label', color='white', fontsize=11)
ax_cm.set_ylabel('True Label', color='white', fontsize=11)
ax_cm.tick_params(colors='white')
ax_cm.set_title('', color='white')

# ── Panel E: Per-class Accuracy ──────────────────────────────
ax_bar = fig.add_subplot(gs[3])
ax_bar.set_facecolor('#1a1a2e')
fig.text(0.5, 0.308, '④ Per-Class Accuracy',
         ha='center', fontsize=13, color='#ffd54f', fontweight='bold')

per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
colors_bar = plt.cm.plasma(np.linspace(0.2, 0.9, 10))
bars = ax_bar.bar(range(10), per_class_acc, color=colors_bar,
                  edgecolor='white', linewidth=0.8)
for bar, acc in zip(bars, per_class_acc):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{acc:.1f}%', ha='center', va='bottom', color='white', fontsize=9)
ax_bar.set_ylim(96, 100.5)
ax_bar.set_xticks(range(10))
ax_bar.set_xticklabels([f'Digit {i}' for i in range(10)], color='white', fontsize=9)
ax_bar.set_ylabel('Accuracy (%)', color='#aaa')
ax_bar.tick_params(colors='#aaa')
for spine in ax_bar.spines.values():
    spine.set_color('#444')

# ── Panel F: Misclassified Examples ─────────────────────────
gs_err = gridspec.GridSpecFromSubplotSpec(1, 10, subplot_spec=gs[4], wspace=0.15)
fig.text(0.5, 0.125, '⑤ Misclassified Samples (True → Predicted)',
         ha='center', fontsize=13, color='#ffd54f', fontweight='bold')

wrong_idx = np.where(y_pred != y_test)[0]
for i in range(min(10, len(wrong_idx))):
    idx = wrong_idx[i]
    ax = fig.add_subplot(gs_err[i])
    ax.imshow(X_test[idx].reshape(28, 28), cmap='hot')
    ax.set_title(f'{y_test[idx]}→{y_pred[idx]}', color='#ff5252', fontsize=9, pad=2)
    ax.axis('off')

import os
output_path = '/mnt/user-data/outputs/project1_digit_recognizer.png'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
plt.savefig(output_path, dpi=150, bbox_inches='tight',
            facecolor='#0d0d0d', edgecolor='none')
plt.close()
print(f"  Visualization saved to: {output_path}")

print("\n" + "=" * 60)
print(f"  ✅ PROJECT 1 COMPLETE")
print(f"  CNN Test Accuracy : {test_accuracy * 100:.2f}%")
print("=" * 60)


[6/6] Generating Visualizations...
  Visualization saved to: /mnt/user-data/outputs/project1_digit_recognizer.png

  ✅ PROJECT 1 COMPLETE
  CNN Test Accuracy : 99.40%
